In [26]:
# =========================================================================
# CELDA MAESTRA: GEOMETRÍA + SIMULACIÓN EN RAM (ELCIGALA 60-CORE MODE)
# =========================================================================

import os
import json
import numpy as np
import subprocess
import shutil
from openEMS import openEMS
from CSXCAD import ContinuousStructure

print("[*] 1. Preparando disco de RAM...")
sim_path = "/work/sim_crosstalk"

if not os.path.exists(sim_path):
    os.makedirs(sim_path)
else:
    # Vaciamos la RAM sin romper el punto de montaje de Docker
    for root, dirs, files in os.walk(sim_path):
        for f in files: os.unlink(os.path.join(root, f))
        for d in dirs: shutil.rmtree(os.path.join(root, d))

print("[*] 2. Cargando geometría...")
with open('/work/experimentos/geometria_completa.json', 'r') as f:
    geo = json.load(f)

unit = 1e-3
thickness = geo['board_params']['thickness'] * unit
er = geo['board_params']['epsilon_r']

# 8000 pasos son ~1.3 nanosegundos. Más que suficiente para ver la explosión del crosstalk.
FDTD = openEMS(EndCriteria=0, NrTS=8000)

CSX = ContinuousStructure()
FDTD.SetCSX(CSX)

all_x = [p['pos_x'] for p in geo['pads']] + [t['start_x'] for t in geo['tracks']] + [t['end_x'] for t in geo['tracks']]
all_y = [p['pos_y'] for p in geo['pads']] + [t['start_y'] for t in geo['tracks']] + [t['end_y'] for t in geo['tracks']]
margin = 5.0 
min_x, max_x = (min(all_x) - margin) * unit, (max(all_x) + margin) * unit
min_y, max_y = (min(all_y) - margin) * unit, (max(all_y) + margin) * unit

FR4 = CSX.AddMaterial('FR4', epsilon=er)
PEC = CSX.AddMetal('PEC') 

print("[*] 3. Dibujando modelo 3D...")
FR4.AddBox([min_x, min_y, 0], [max_x, max_y, thickness], priority=0)
PEC.AddBox([min_x, min_y, 0], [max_x, max_y, 0], priority=10)

def get_thick_line_pts(x1, y1, x2, y2, w):
    dx, dy = x2 - x1, y2 - y1
    L = np.sqrt(dx**2 + dy**2)
    if L == 0: return [[x1-w/2, x1+w/2, x1+w/2, x1-w/2], [y1-w/2, y1-w/2, y1+w/2, y1+w/2]]
    nx, ny = -dy / L * (w/2), dx / L * (w/2)
    return [[x1+nx, x1-nx, x2-nx, x2+nx], [y1+ny, y1-ny, y2-ny, y2+ny]]

for track in geo['tracks']:
    w = track['width'] * unit
    pts = get_thick_line_pts(track['start_x']*unit, track['start_y']*unit, 
                             track['end_x']*unit, track['end_y']*unit, w)
    PEC.AddPolygon(pts, 'z', thickness, priority=20)

for pad in geo['pads']:
    sx, sy = (pad['size_x'] * unit) / 2, (pad['size_y'] * unit) / 2
    px, py = pad['pos_x'] * unit, pad['pos_y'] * unit
    PEC.AddBox([px-sx, py-sy, thickness], [px+sx, py+sy, thickness], priority=20)

print("[*] 4. Configurando malla...")
mesh = CSX.GetGrid()
mesh.SetDeltaUnit(1)
res_pista = 0.15 * unit 
mesh.AddLine('x', [min_x, max_x] + [p['pos_x']*unit for p in geo['pads']])
mesh.AddLine('y', [min_y, max_y] + [p['pos_y']*unit for p in geo['pads']])
mesh.AddLine('z', [0, thickness, thickness + 2*unit])
mesh.SmoothMeshLines('x', res_pista, ratio=1.5)
mesh.SmoothMeshLines('y', res_pista, ratio=1.5)
mesh.SmoothMeshLines('z', 0.1 * unit, ratio=1.5)

print("[*] 5. Configurando excitación TANH y Puertos...")
f_max = 1e9 

# --- MEJORA 1: Adelantamos el disparo a 0.2ns para no simular "silencio" inútil ---
formula_escalon = "5 * (1 + tanh((t - 0.2e-9) / 0.1e-9))"
FDTD.SetCustomExcite(formula_escalon.encode('ascii'), 0, f_max)
FDTD.SetBoundaryCond([8, 8, 8, 8, 8, 8])

pad_ag_x, pad_ag_y = 113.41 * unit, 80.46 * unit
pad_vic_x, pad_vic_y = 113.41 * unit, 83.0 * unit
FDTD.AddLumpedPort(1, 50.0, [pad_ag_x, pad_ag_y, 0], [pad_ag_x, pad_ag_y, thickness], 'z', excite=10.0, priority=50)
FDTD.AddLumpedPort(2, 1e6, [pad_vic_x, pad_vic_y, 0], [pad_vic_x, pad_vic_y, thickness], 'z', excite=0.0, priority=50)

print("[*] 6. Configurando capturas 2D ultraligeras...")
et_dump = CSX.AddDump('E_Field_Heatmap', dump_type=0)

# Muestreo espacial agresivo (Saltamos 4 celdas). El mapa de calor se verá igual de bien.
et_dump.SetSubSampling([4, 4, 1])

# --- MEJORA 2: Cámara estrictamente 2D ---
# Al poner Z=thickness en ambos lados, OpenEMS no calcula volumen, solo copia el plano exacto del cobre.
et_dump.AddBox([min_x, min_y, thickness], [max_x, max_y, thickness], priority=0)

xml_file = os.path.join(sim_path, 'estructura.xml')
FDTD.Write2XML(xml_file)

# --- 1. EL PARCHE XML CORRECTO PARA EL VÍDEO ---
import xml.etree.ElementTree as ET
import os
import subprocess

xml_file = os.path.join(sim_path, 'estructura.xml')
print("[*] Aplicando parche XML para grabar frames de vídeo...")

tree = ET.parse(xml_file)
root = tree.getroot()

# Buscamos la etiqueta de la cámara y modificamos su sub-nodo <DumpBox>
for prop in root.findall(".//Property[@name='E_Field_Heatmap']"):
    dumpbox = prop.find('DumpBox')
    if dumpbox is not None:
        dumpbox.set('Decimation', '100') # Graba 1 frame cada 100 cálculos de física

tree.write(xml_file)

# --- 2. LANZAMIENTO SIN INVENTOS (BASH PURO) ---
print("=========================================")
print(f"[*] LANZANDO EL CIGALA")
print("=========================================")

# Forzamos las variables de entorno directamente en la línea de comandos de Linux
comando_bash = "export OMP_NUM_THREADS=30 && export OMP_DYNAMIC=FALSE && /opt/openEMS/bin/openEMS estructura.xml"

process = subprocess.Popen(
    comando_bash,
    cwd=sim_path,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

try:
    for line in process.stdout:
        print(line, end="")
except Exception as e:
    print(f"\n❌ Error leyendo la salida: {e}")

process.wait()

# AHORA SÍ comprobamos si ha fallado de verdad
if process.returncode == 0:
    print("\n✅ SIMULACIÓN EXITOSA DE VERDAD.")
else:
    print(f"\n❌ ERROR: El motor crasheó (Código {process.returncode}). Revisa las líneas de arriba.")

[*] 1. Preparando disco de RAM...
[*] 2. Cargando geometría...
[*] 3. Dibujando modelo 3D...
[*] 4. Configurando malla...
[*] 5. Configurando excitación TANH y Puertos...
[*] 6. Configurando capturas 2D ultraligeras...
[*] Aplicando parche XML para grabar frames de vídeo...
[*] LANZANDO EL CIGALA
 ---------------------------------------------------------------------- 
 | openEMS 64bit -- version v0.0.36-142-g32c5c6b
 | (C) 2010-2025 Thorsten Liebig <thorsten.liebig@gmx.de>  GPL license
 ---------------------------------------------------------------------- 
	Used external libraries:
		CSXCAD -- Version: v0.6.3-100-gd7d70ef
		hdf5   -- Version: 1.10.10
		          compiled against: HDF5 library version: 1.10.10
		tinyxml -- compiled against: 2.6.2
		fparser
		boost  -- compiled against: 1_83
		vtk -- Version: 9.1.0
		       compiled against: 9.1.0

Create FDTD operator (compressed SSE + multi-threading)
FDTD simulation size: 364x113x47 --> 1.9332e+06 FDTD cells 
FDTD timestep is: 1.6032

In [8]:
sim_path

'/work/sim_crosstalk'

In [21]:
import os
import numpy as np
import matplotlib.pyplot as plt

def leer_datos_openems(ruta_sim, num):
    # El nombre exacto que generó el motor en tu carpeta
    archivo = os.path.join(ruta_sim, f'port_ut_{num}')
    
    if not os.path.exists(archivo):
        raise FileNotFoundError(f"No se encuentra el archivo: {archivo}")
        
    # Cargamos el texto saltando el encabezado '%' 
    # Columnas: [Tiempo, Parte_Real, Parte_Imaginaria]
    data = np.loadtxt(archivo, comments='%')
    return data[:, 0], data[:, 1]

try:
    # 1. Leer los datos
    t_segundos, v_agresor = leer_datos_openems(sim_path, 1)
    _, v_victima = leer_datos_openems(sim_path, 2)
    t_ns = t_segundos * 1e9

    # 2. Configurar la gráfica con Doble Eje Y
    fig, ax1 = plt.subplots(figsize=(12, 7))

    # --- EJE IZQUIERDO (Agresor - Voltios) ---
    color_ag = 'tab:red'
    ax1.set_xlabel('Tiempo (ns)', fontsize=12)
    ax1.set_ylabel('Señal Agresora (Voltios)', color=color_ag, fontsize=12)
    ax1.plot(t_ns, v_agresor, color=color_ag, linewidth=2.5, label='Agresor (10V Step)')
    ax1.tick_params(axis='y', labelcolor=color_ag)
    ax1.set_ylim(-2, 12) # Para ver bien el paso de 0 a 10
    ax1.grid(True, linestyle=':', alpha=0.6)

    # --- EJE DERECHO (Víctima - Milivoltios) ---
    ax2 = ax1.twinx() 
    color_vic = 'tab:blue'
    ax2.set_ylabel('Ruido Inducido (mV)', color=color_vic, fontsize=12)
    ax2.plot(t_ns, v_victima * 1000, color=color_vic, linewidth=2, label='Víctima (Crosstalk)')
    ax2.tick_params(axis='y', labelcolor=color_vic)
    # Ajustamos el rango de la víctima según el pico detectado (aprox +/- 5mV)
    ax2.set_ylim(-10, 10) 

    # Título y Leyendas
    plt.title("Respuesta al Escalón: Sincronía Agresor vs Víctima", fontsize=14, fontweight='bold')
    
    # Combinar las leyendas de ambos ejes en una sola caja
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.tight_layout()
    plt.show()

    # 3. Resultado numérico
    max_crosstalk = np.max(np.abs(v_victima)) * 1000
    print(f"\n✅ ANÁLISIS COMPLETADO")
    print(f"--------------------------------------------------")
    print(f"Pico máximo de ruido inducido (Crosstalk): {max_crosstalk:.2f} mV")
    print(f"--------------------------------------------------")

except Exception as e:
    print(f"❌ Error al procesar los resultados: {e}")

❌ Error al procesar los resultados: too many indices for array: array is 1-dimensional, but 2 were indexed


In [23]:
import pyvista as pv
import os
import numpy as np

sim_path = "/work/sim_crosstalk"
video_output = "/work/crosstalk_heatmap.mp4" 
field_dump_name = "E_Field_Heatmap"

# 1. Buscamos TODOS los archivos
todas_las_fotos = sorted([f for f in os.listdir(sim_path) if f.startswith(field_dump_name) and f.endswith('.vtr')],
               key=lambda x: int(x.split('_')[-1].split('.')[0]))

# 2. Nos quedamos solo con 1 de cada 50 para que el vídeo vaya a buena velocidad
files = todas_las_fotos[::50] 

print(f"✅ Encontrados {len(todas_las_fotos)} pasos de tiempo.")
print(f"[*] Renderizando {len(files)} frames para el vídeo...")

plotter = pv.Plotter(off_screen=True, window_size=[1280, 720])
plotter.set_background('white')

first_mesh = pv.read(os.path.join(sim_path, files[0]))
first_mesh['E_Mag'] = np.linalg.norm(first_mesh['E-Field'], axis=1)

heatmap = plotter.add_mesh(first_mesh, scalars='E_Mag', cmap='inferno', 
                          clim=[0, 3e3],
                          scalar_bar_args={'title': 'Campo E (V/m)'},
                          show_edges=False)

plotter.add_text("Tiempo: 0.00 ns", position='upper_left', color='black', font_size=12, name="reloj")
plotter.camera_position = 'xy'
plotter.reset_camera()
plotter.camera.zoom(1.2)

plotter.open_movie(video_output, framerate=24) # 24 fps para un look de cine

for i, filename in enumerate(files):
    if i % 10 == 0: print(f"   Renderizando frame {i}/{len(files)}...")
        
    mesh_t = pv.read(os.path.join(sim_path, filename))
    mesh_t['E_Mag'] = np.linalg.norm(mesh_t['E-Field'], axis=1)
    heatmap.mapper.dataset.copy_from(mesh_t)
    
    # Tiempo estimado: 6000 pasos * 1.6e-13s por paso = ~1.0 ns
    t_ns = (i / len(files)) * 1.0 
    plotter.add_text(f"Tiempo estim: {t_ns:.2f} ns", position='upper_left', color='black', font_size=12, name="reloj")
    
    plotter.write_frame()

plotter.close()
print(f"✅ VÍDEO GENERADO: {video_output}")

✅ Encontrados 1 pasos de tiempo.
[*] Renderizando 1 frames para el vídeo...
   Renderizando frame 0/1...
✅ VÍDEO GENERADO: /work/crosstalk_heatmap.mp4


In [24]:
!tree /work

/work
├── Dockerfile
├── crosstalk_heatmap.mp4
├── docker-compose.yml
├── experimentos
│   ├── geometria_completa.json
│   └── openems_crosstalk.ipynb
└── sim_crosstalk
    ├── E_Field_Heatmap_0000000000.vtr
    ├── estructura.xml
    ├── et
    ├── ht
    ├── port_it_1
    ├── port_it_2
    ├── port_ut_1
    └── port_ut_2

3 directories, 13 files
